In [1]:
!pip install -U langchain langchain-core langchain-community langchain-google-genai streamlit pyngrok langserve fastapi uvicorn ollama sse_starlette

In [2]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:7 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,514 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,843 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,563 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,184 kB]
Get:14 https://r2u.

In [3]:
import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

Starting Ollama server...


In [4]:
print("Downloading llama2 model (this may take a few minutes)...")
!ollama pull llama2
print("Done! Ollama is now ready to serve queries.")


Done! Ollama is now ready to serve queries.


In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "<ngrok-authtoken>"

ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [32]:
%%writefile app.py

from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langserve import add_routes
import os
from langchain_community.llms import Ollama

os.environ['GEMINI_API_KEY'] = "<api-key>"

app = FastAPI(
    title="LangChain Server",
    version="1.0",
    description="A Simple API Server"
)

# Configure Gemini model with a valid model name
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
llm = Ollama(model="llama2")

# Construct Prompt Templates using standard curly braces
prompt1 = ChatPromptTemplate.from_template("Write me an essay about {topic} in approx 100 words")
prompt2 = ChatPromptTemplate.from_template("Write me a poem about {topic} in approx 100 words")

# Expose API endpoints
add_routes(
    app,
    prompt1 | model,
    path="/essay"
)

add_routes(
    app,
    prompt2 | llm,
    path="/poem"
)


Overwriting app.py


In [25]:
import subprocess
import time
from pyngrok import ngrok

# Terminate any existing servers and Ngrok tunnels
subprocess.run("pkill -f uvicorn", shell=True)
subprocess.run("pkill -f streamlit", shell=True)
ngrok.kill()

# Start the FastAPI LangServe server on port 8000
print("Starting backend server on port 8000...")
subprocess.Popen(["uvicorn", "app:app", "--port", "8000", "--host", "0.0.0.0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Create Ngrok Tunnel for the FastAPI backend (8000)
backend_tunnel = ngrok.connect(8000, "http")
# Extract public URL (replace 'http' with 'https' to avoid mixed-content/insecure blocks)
backend_url = backend_tunnel.public_url.replace("http://", "https://")
print(f"Backend API Server Tunnel is active: {backend_url}")


Starting backend server on port 8000...
Backend API Server Tunnel is active: https://surfacing-parakeet-immerse.ngrok-free.dev


In [26]:
%%writefile client.py

import requests
import streamlit as st

def get_gemini_response(input_text):
  response = requests.post(
      "https://surfacing-parakeet-immerse.ngrok-free.dev/essay/invoke",
      json = {'input': {'topic' : input_text}}
  )
  return response.json()['output']['content']

def get_ollama_response(input_text):
  response = requests.post(
      "https://surfacing-parakeet-immerse.ngrok-free.dev/poem/invoke",
      json = {'input': {'topic' : input_text}}
  )
  return response.json()['output']

st.title("LangChain demo with GEMINI+LLAMA2 API")
input_text1 = st.text_input("Write an essay on...") # Gemini
input_text2 = st.text_input("Write an poem on...") #llama2

if input_text1:
  st.write(get_gemini_response(input_text1))

if input_text2:
  st.write(get_ollama_response(input_text2))

Overwriting client.py


In [27]:
# Start Streamlit app
print("Launching Streamlit on port 8501...")
subprocess.Popen(["streamlit", "run", "client.py", "--server.port", "8501", "--server.address", "0.0.0.0"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Expose Streamlit via Ngrok
streamlit_tunnel = ngrok.connect(8501, "http")
frontend_url = streamlit_tunnel.public_url.replace("http://", "https://")
print(f"Streamlit App is live at: {frontend_url}")


Launching Streamlit on port 8501...
Streamlit App is live at: https://surfacing-parakeet-immerse.ngrok-free.dev
